In [0]:
pip install python-jobspy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 17.1 MB/s eta 0:00:00
  Attempting uninstall: NUMPY
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-42320ed1-a96a-4777-aa4f-3ead4712d761
    Can't uninstall 'numpy'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
LinkedIn + Indeed Tech Jobs Scraper
====================================
Scrapes tech job listings from LinkedIn and Indeed (in a single combined
search per keyword/location) across major Arab and foreign countries.
Only jobs posted within the last 24 hours are collected (hours_old=24).
Handles pagination, retries, deduplication, and checkpointing automatically.

Output: overwrites the Databricks Delta table
        depi_project.philo_files.bronze_linkedin_indeed

Usage:
    pip install python-jobspy pandas tqdm
    python linkedin_indeed_scraper.py
"""

import pandas as pd
import time
import random
import os
from tqdm import tqdm

try:
    from jobspy import scrape_jobs
except ImportError:
    print("ERROR: jobspy is not installed. Run:")
    print("   pip install python-jobspy")
    exit(1)


# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

# Job titles to search for across all locations
KEYWORDS = [
    "software engineer",
    "frontend developer",
    "backend developer",
    "full stack developer",
    "mobile developer",
    "ios developer",
    "android developer",
    "web developer",
    # Data & AI
    "data scientist",
    "data analyst",
    "data engineer",
    "machine learning engineer",
    "AI engineer",
    "business intelligence",
    "nlp engineer",
    "computer vision engineer",
    # Infrastructure & Security
    "devops engineer",
    "cloud engineer",
    "site reliability engineer",
    "cybersecurity engineer",
    "network engineer",
    "linux administrator",
    # Product & Design
    "product manager",
    "product designer",
    "UX designer",
    "UI designer",
    "UX researcher",
    # Other Tech
    "QA engineer",
    "blockchain developer",
    "game developer",
]

# Target countries — key is the display name, value holds the location
# string used by LinkedIn and the country string used by Indeed (needed
# because a single combined scrape_jobs() call still requires both
# per-site location formats).
LOCATIONS = {
    # Arab countries
    "Egypt":          {"location": "Egypt",                "country_indeed": "egypt"},
    "Saudi Arabia":   {"location": "Saudi Arabia",          "country_indeed": "saudi arabia"},
    "UAE":            {"location": "United Arab Emirates",  "country_indeed": "united arab emirates"},
    "Qatar":          {"location": "Qatar",                 "country_indeed": "qatar"},
    "Kuwait":         {"location": "Kuwait",                "country_indeed": "kuwait"},
}

# Number of results to fetch per keyword/location combination
RESULTS_PER_SEARCH = 20

# Only fetch jobs posted within the last 24 hours
HOURS_OLD = 24

# Random delay between requests to avoid rate limiting (in seconds)
DELAY_MIN = 8
DELAY_MAX = 15

# Checkpoint file — stored in a Unity Catalog Volume inside depi_project
# so it lives alongside the output table rather than in workspace files.
CATALOG       = "depi_project"
SCHEMA        = "philo_files"
VOLUME        = "checkpoints"
VOLUME_PATH   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CHECKPOINT_FILE = f"{VOLUME_PATH}/linkedin_indeed_checkpoint.csv"

# Destination Delta table for the final output
DELTA_TABLE = f"{CATALOG}.{SCHEMA}.bronze_linkedin_indeed"


def ensure_volume_exists():
    """Create the checkpoint volume if it doesn't already exist."""
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")


# ─────────────────────────────────────────────
# Columns to keep in the final output
# ─────────────────────────────────────────────

COLUMNS = [
    "title",       # Job title
    "company",     # Company name
    "location",    # City / region
    "country",     # Country (added by us)
    "date_posted", # Posting date
    "job_type",    # Full-time, part-time, remote, etc.
    "job_url",     # Direct link to the job posting
    "source",      # Platform the job was scraped from (linkedin / indeed)
]


# ─────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────

def load_checkpoint():
    """Load previously saved progress so we can resume if the script was interrupted."""
    if os.path.exists(CHECKPOINT_FILE):
        print("Checkpoint found. Resuming from last saved point...")
        return pd.read_csv(CHECKPOINT_FILE)
    return pd.DataFrame()


def save_checkpoint(df):
    """Save current progress to a temporary CSV file."""
    df.to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")


def clean_dataframe(df, country_name):
    """
    Normalize and clean a scraped dataframe:
    - Keep only the columns we need
    - Add the country column
    - Add the source column (linkedin / indeed), derived from JobSpy's
      'site' field
    - Standardize date format
    - Strip whitespace from text fields
    """
    # JobSpy tags each row with the platform it came from in a "site"
    # column — capture it as "source" before filtering down to COLUMNS
    if "site" in df.columns:
        df = df.copy()
        df["source"] = df["site"]

    # Keep only available columns from our list
    available = [c for c in COLUMNS if c in df.columns]
    df = df[available].copy()

    # Tag each row with the country it was scraped from
    df["country"] = country_name

    # Standardize date to YYYY-MM-DD
    if "date_posted" in df.columns:
        df["date_posted"] = pd.to_datetime(
            df["date_posted"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    # Strip extra whitespace from text columns
    for col in ["title", "company", "location", "source"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    return df


def scrape_with_retry(keyword, location_name, country_code, retries=3):
    """
    Scrape LinkedIn + Indeed together for a given keyword and location,
    limited to jobs posted in the last HOURS_OLD hours.
    Retries up to 3 times with increasing wait time on failure.
    Returns an empty DataFrame if all attempts fail.
    """
    for attempt in range(1, retries + 1):
        try:
            jobs = scrape_jobs(
                site_name=["linkedin", "indeed"],
                search_term=keyword,
                location=location_name,
                country_indeed=country_code,
                results_wanted=RESULTS_PER_SEARCH,
                hours_old=HOURS_OLD,
            )
            return jobs
        except Exception as e:
            wait = attempt * 10
            print(f"      Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                print(f"      Waiting {wait} seconds before retry...")
                time.sleep(wait)

    # Return empty DataFrame if all retries failed
    return pd.DataFrame()


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    print("=" * 55)
    print("  LinkedIn + Indeed Tech Jobs Scraper (last 24h)")
    print(f"  {len(KEYWORDS)} keywords x {len(LOCATIONS)} locations")
    print(f"  Total searches: {len(KEYWORDS) * len(LOCATIONS)}")
    print("=" * 55)

    # Make sure the Unity Catalog volume for checkpoints exists
    ensure_volume_exists()

    # Load checkpoint data if available
    all_jobs = load_checkpoint()

    # Track already seen URLs to avoid duplicates across searches
    seen_urls = set(all_jobs["job_url"].tolist()) if not all_jobs.empty else set()

    total_searches = len(KEYWORDS) * len(LOCATIONS)
    search_count   = 0
    new_jobs_total = 0

    # Progress bar to track overall scraping progress
    pbar = tqdm(total=total_searches, desc="Overall Progress", unit="search")

    for location_name, loc_info in LOCATIONS.items():
        for keyword in KEYWORDS:
            search_count += 1

            # Update progress bar label
            pbar.set_postfix({
                "location": location_name[:10],
                "keyword":  keyword[:15],
                "total":    len(all_jobs),
            })

            # Scrape jobs (LinkedIn + Indeed) for this keyword + location
            df = scrape_with_retry(
                keyword,
                loc_info["location"],
                loc_info["country_indeed"],
            )

            if not df.empty:
                df = clean_dataframe(df, location_name)

                # Remove jobs we've already collected
                if "job_url" in df.columns:
                    df = df[~df["job_url"].isin(seen_urls)]
                    seen_urls.update(df["job_url"].tolist())

                if not df.empty:
                    new_jobs_total += len(df)
                    all_jobs = pd.concat([all_jobs, df], ignore_index=True)

            # Save checkpoint every 10 searches in case of interruption
            if search_count % 10 == 0 and not all_jobs.empty:
                save_checkpoint(all_jobs)
                tqdm.write(f"  Checkpoint saved — {len(all_jobs)} jobs so far")

            pbar.update(1)

            # Random delay to reduce the chance of getting blocked
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    pbar.close()

    # ─────────────────────────────────────────
    # Save final output
    # ─────────────────────────────────────────

    if all_jobs.empty:
        print("\nNo data collected.")
        return

    # Final deduplication by job URL
    if "job_url" in all_jobs.columns:
        before = len(all_jobs)
        all_jobs = all_jobs.drop_duplicates(subset=["job_url"])
        removed = before - len(all_jobs)
        if removed:
            print(f"\nRemoved {removed} duplicate entries.")

    # Sort by most recent postings first
    if "date_posted" in all_jobs.columns:
        all_jobs = all_jobs.sort_values("date_posted", ascending=False)

    # Write final output by overwriting the Databricks Delta table
    spark.createDataFrame(all_jobs).write.format("delta").mode("overwrite").saveAsTable(DELTA_TABLE)

    # Clean up checkpoint file after successful completion
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    # ─────────────────────────────────────────
    # Print summary
    # ─────────────────────────────────────────

    print("\n" + "=" * 55)
    print(f"  Done! Output written to table: {DELTA_TABLE}")
    print(f"  Total jobs collected: {len(all_jobs):,}")
    print(f"  New unique jobs: {new_jobs_total:,}")

    print("\n  Jobs by country:")
    if "country" in all_jobs.columns:
        for country, count in all_jobs["country"].value_counts().items():
            print(f"    {country:<20} {count:>5,}")

    print("\n  Jobs by source:")
    if "source" in all_jobs.columns:
        for source, count in all_jobs["source"].value_counts().items():
            print(f"    {source:<20} {count:>5,}")

    print("\n  Top job titles:")
    if "title" in all_jobs.columns:
        for title, count in all_jobs["title"].value_counts().head(5).items():
            print(f"    {title:<35} {count:>4,}")

    print("=" * 55)


main()

  LinkedIn + Indeed Tech Jobs Scraper (last 24h)
  1 keywords x 5 locations
  Total searches: 5


Overall Progress: 100%|██████████| 5/5 [01:31<00:00, 18.40s/search, location=Kuwait, keyword=software engine, total=107]



  Done! Output written to table: depi_project.philo_files.bronze_linkedin_indeed
  Total jobs collected: 112
  New unique jobs: 112

  Jobs by country:
    UAE                     40
    Saudi Arabia            31
    Egypt                   25
    Qatar                   11
    Kuwait                   5

  Jobs by source:
    indeed                  57
    linkedin                55

  Top job titles:
    Remote Software Engineer (Go/Golang)    2
    Senior Public Health BIM Engineer      2
    Associate Designer II                  2
    Mechanical Inspector                   2
    Early Careers -Geoscience&Petrotechnical    2
